# Hypothesis 6 — Main Notebook

This notebook prepares three sets of 34 experiment configurations (English, Spanish, Mandarin),
runs them selectively in parallel, and compares outcome distributions across languages.

Requirements implemented:
- 34 configs per language (total 102) with agent temperatures drawn from U(0, 1.5) per config.
- Separate subfolders per language under `configs/`, `terminal_outputs/`, and `results/`.
- Statistical analysis comparing the outcomes across the three language sets (5×3 table).

In [2]:
# Imports
import sys, os
from pathlib import Path

# Ensure repo root on sys.path (for local package imports)
def _add_repo_root_to_sys_path():
    here = Path.cwd().resolve()
    for p in [here] + list(here.parents):
        if (p / 'main.py').exists() and (p / 'hypothesis_testing').is_dir():
            if str(p) not in sys.path:
                sys.path.insert(0, str(p))
            return p
    return here
_REPO_ROOT = _add_repo_root_to_sys_path()

import json
import random
import shutil
import yaml
import numpy as np
from collections import Counter
from scipy.stats import chi2_contingency

from hypothesis_testing.utils_hypothesis_testing.runner import (
    list_config_files,
    select_configs,
    run_configs_in_parallel,
)


In [ ]:
# Base paths and per-language subfolders
BASE_DIR = _REPO_ROOT / 'hypothesis_testing' / 'hypothesis_6'
CONFIGS_BASE = BASE_DIR / 'configs'
LOGS_BASE = BASE_DIR / 'terminal_outputs'
RESULTS_BASE = BASE_DIR / 'results'
TRANSCRIPTS_BASE = BASE_DIR / 'transcripts'

LANG_SETS = {
    'english': 'English',
    'spanish': 'Spanish',
    'mandarin': 'Mandarin',
}

# Ensure subfolders exist (does not create config files)
for key in LANG_SETS.keys():
    (CONFIGS_BASE / key).mkdir(parents=True, exist_ok=True)
    (LOGS_BASE / key).mkdir(parents=True, exist_ok=True)
    (RESULTS_BASE / key).mkdir(parents=True, exist_ok=True)
    (TRANSCRIPTS_BASE / key).mkdir(parents=True, exist_ok=True)

CONFIGS_BASE, LOGS_BASE, RESULTS_BASE, TRANSCRIPTS_BASE, LANG_SETS


## 1) Config Generation 

Generates 34 YAML configurations for each language set.
- All 5 agents share a per-config temperature drawn from U(0, 1.5).


In [ ]:
# Income class probabilities (must sum to 1.0)
INCOME_CLASS_PROBS = {
    'high': 0.05,
    'medium_high': 0.10,
    'medium': 0.50,
    'medium_low': 0.25,
    'low': 0.10,
}

# Placeholder model list for participant agents — adjust as needed
MODEL_LIST = [
    "gemini-2.5-pro",
    "gemini-2.5-flash",
    "gemini-2.5-flash-lite",
    "google/gemma-3-27b-it",
]

def make_agents_with_models(temp: float, models: list[str]) -> list[dict]:
    agents = []
    for i in range(0, 5):  # 5 participant agents
        agents.append({
            'name': f'Agent_{i}',
            'personality': 'You are an American college student',
            'model': models[i],
            'temperature': float(temp),
            'memory_character_limit': 25000,
            'reasoning_enabled': True,
        })
    return agents

def build_config(lang: str, temp: float, seed_val: int, models: list[str]) -> dict:
    return {
        'language': lang,
        'seed': int(seed_val),
        'agents': make_agents_with_models(temp, models),
        'utility_agent_model': "gemini-2.5-flash-lite",
        'utility_agent_temperature': 0.0,
        'phase2_rounds': 10,
        'distribution_range_phase2': [2, 6],
        'income_class_probabilities': INCOME_CLASS_PROBS,
        'original_values_mode': { 'enabled': True },
    }

# Optional: set a global seed for reproducible generation (adjust or comment out)
GLOBAL_SEED = 10000
random.seed(GLOBAL_SEED)
np.random.seed(GLOBAL_SEED)

def generate_aligned_configs(n: int = 34) -> dict[str, list[Path]]:
    paths: dict[str, list[Path]] = {k: [] for k in LANG_SETS.keys()}
    for idx in range(1, n + 1):
        temp = random.uniform(0.0, 1.5)  # shared per condition across languages
        seed_val = random.randint(0, 2**31 - 1)
        models = [random.choice(MODEL_LIST) for _ in range(5)]  # shared agent models
        for lang_key, lang_name in LANG_SETS.items():
            cfg = build_config(lang=lang_name, temp=temp, seed_val=seed_val, models=models)
            out_dir = (CONFIGS_BASE / lang_key)
            out_dir.mkdir(parents=True, exist_ok=True)
            fname = out_dir / f'hypothesis_6_{lang_key}_condition_{idx}_config.yaml'
            with open(fname, 'w') as f:
                yaml.safe_dump(cfg, f, sort_keys=False)
            paths[lang_key].append(fname)
    return paths

# To generate aligned configs for all languages at once (writes 102 files total):
# files_by_lang = generate_aligned_configs(n=34)
# {k: len(v) for k, v in files_by_lang.items()}


## 2) Run Configs (Parallel per Language)

Select subsets and run with per-language logs/results directories.

In [ ]:
def run_language_set(lang_key: str, include_indices=None, include_names=None, concurrency: int = 4, timeout_sec: int | None = None):
    cfg_dir = CONFIGS_BASE / lang_key
    logs_dir = LOGS_BASE / lang_key
    results_dir = RESULTS_BASE / lang_key
    configs = list_config_files(cfg_dir)

    selected = select_configs(configs, include_indices=include_indices, include_names=include_names)
    print(f'[{lang_key}] Found {len(configs)} configs; selected {len(selected)}')

    run_results = run_configs_in_parallel(
        selected,
        concurrency=concurrency,
        logs_dir=logs_dir,
        results_dir=results_dir,
        timeout_sec=timeout_sec,
    )
    ok = sum(1 for r in run_results if r.get('ok'))
    print(f'[{lang_key}] Completed: {ok}/{len(run_results)} OK')
    return run_results

# Example usage (uncomment to run a small sample from each set):
# rr_en = run_language_set('english', include_indices=[1,2,3], concurrency=3, timeout_sec=None)
# rr_es = run_language_set('spanish', include_indices=[1,2,3], concurrency=3, timeout_sec=None)
# rr_zh = run_language_set('mandarin', include_indices=[1,2,3], concurrency=3, timeout_sec=None)


## 3) Analysis — Compare Outcomes Across Languages

Build a 5×3 contingency table (rows=principle/disagreement categories; columns=English/Spanish/Mandarin)
and run Fisher–Freeman–Halton exact test via R when available.
Also compute Cramér's V with optional bootstrap CI.

In [ ]:
CATEGORIES = [
    'maximizing_floor',
    'maximizing_average',
    'maximizing_average_floor_constraint',
    'maximizing_average_range_constraint',
    'disagreement',
]

def categorize_result(result_path: Path) -> str:
    try:
        with open(result_path, 'r') as f:
            data = json.load(f)
        gi = data.get('general_information', {})
        consensus = gi.get('consensus_reached', False)
        principle = gi.get('consensus_principle')
        if consensus and principle in CATEGORIES:
            return principle
        return 'disagreement'
    except Exception:
        return 'disagreement'

def count_by_language() -> dict[str, Counter]:
    out: dict[str, Counter] = {}
    for k in LANG_SETS.keys():
        counts = Counter()
        result_files = sorted((RESULTS_BASE / k).glob('*_results.json'))
        for rp in result_files:
            counts[categorize_result(rp)] += 1
        for cat in CATEGORIES:
            counts.setdefault(cat, 0)
        out[k] = counts
    return out

lang_counts = count_by_language()
for k, counts in lang_counts.items():
    print(f'{k.capitalize()} counts:', dict(counts))

# Build contingency table: rows=categories, cols=[English, Spanish, Mandarin]
col_order = ['english', 'spanish', 'mandarin']
contingency = np.vstack([[lang_counts[col][cat] for col in col_order] for cat in CATEGORIES])
contingency, CATEGORIES, col_order


In [ ]:
def fisher_freeman_halton_pvalue_r(contingency: np.ndarray) -> float | None:
    """Run Fisher–Freeman–Halton test via R's fisher.test if available.
    Returns p-value or None if Rscript not found or fails.
    """
    if shutil.which('Rscript') is None:
        return None
    r_matrix = ','.join(str(int(x)) for x in contingency.flatten(order='C'))
    nrow, ncol = contingency.shape
    r_code = f"""m <- matrix(c({r_matrix}), nrow={nrow}, ncol={ncol}, byrow=TRUE);
f <- tryCatch(fisher.test(m), error=function(e) NA);
if (is.list(f)) {{ cat(f$p.value) }} else {{ cat('NA') }}
"""
    import subprocess
    try:
        out = subprocess.check_output(['Rscript', '-e', r_code], stderr=subprocess.STDOUT, text=True)
        out = out.strip()
        return float(out) if out and out != 'NA' else None
    except Exception:
        return None

p_ffh = fisher_freeman_halton_pvalue_r(contingency)
if p_ffh is None:
    print('R not available; skipping Fisher–Freeman–Halton exact test')
else:
    print(f'Fisher–Freeman–Halton exact test p-value: {p_ffh:.6f}')


In [ ]:
def cramers_v(contingency: np.ndarray) -> float:
    chi2, _, _, _ = chi2_contingency(contingency)
    n = contingency.sum()
    r, c = contingency.shape
    return float(np.sqrt((chi2 / n) / (min(r - 1, c - 1))))

def bias_corrected_cramers_v(contingency: np.ndarray) -> float:
    # Bergsma (2013) bias correction
    chi2, _, _, _ = chi2_contingency(contingency)
    n = contingency.sum()
    r, c = contingency.shape
    phi2 = chi2 / n
    r1 = r - 1
    c1 = c - 1
    phi2_corr = max(0.0, phi2 - (r1 * c1) / (n - 1))
    r_corr = r - ((r - 1) ** 2) / (n - 1)
    c_corr = c - ((c - 1) ** 2) / (n - 1)
    denom = min(r_corr - 1, c_corr - 1)
    if denom <= 0:
        return 0.0
    return float(np.sqrt(phi2_corr / denom))

def bootstrap_cramers_v(contingency: np.ndarray, n_bootstrap: int = 2000, confidence_level: float = 0.95, bias_corrected: bool = True, seed: int | None = 123) -> tuple[np.ndarray, float, float]:
    rng = np.random.default_rng(seed)
    n = int(contingency.sum())
    r, c = contingency.shape
    # Expand counts to individual pairs
    pairs = []
    for i in range(r):
        for j in range(c):
            pairs.extend([(i, j)] * int(contingency[i, j]))
    pairs = np.array(pairs)
    if len(pairs) == 0:
        print('No data available for bootstrap.')
        vs = np.array([0.0])
        return vs, 0.0, 0.0
    vs = []
    for _ in range(n_bootstrap):
        idx = rng.integers(0, len(pairs), size=n)
        sample = pairs[idx]
        # Re-tabulate
        tab = np.zeros_like(contingency)
        for (i, j) in sample:
            tab[i, j] += 1
        v = bias_corrected_cramers_v(tab) if bias_corrected else cramers_v(tab)
        vs.append(v)
    vs = np.array(vs)
    alpha = 1 - confidence_level
    lo, hi = np.quantile(vs, [alpha / 2, 1 - alpha / 2])
    return vs, float(lo), float(hi)

# Effect size summary
if contingency.sum() > 0:
    v_std = cramers_v(contingency)
    v_corr = bias_corrected_cramers_v(contingency)
    print(f"Cramér's V (standard): {v_std:.4f}")
    print(f"Cramér's V (bias-corrected): {v_corr:.4f}")
    # Bootstrap CI (adjust n_bootstrap for speed if needed)
    boot_vs, ci_lo, ci_hi = bootstrap_cramers_v(contingency, n_bootstrap=2000, confidence_level=0.95, bias_corrected=True, seed=123)
    print(f"95% CI for bias-corrected V: [{ci_lo:.4f}, {ci_hi:.4f}]")
else:
    print('Insufficient data for effect size computation')
